# Lora单卡训练 

In [12]:
import os
import json
import torch
from datasets import Dataset
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, DataCollatorForSeq2Seq, TrainingArguments, Trainer, EarlyStoppingCallback
from peft import LoraConfig, TaskType, get_peft_model
from modelscope import snapshot_download

In [13]:
traindata_path = "../datasets/11/train_test/train0819.jsonl"
evaldata_path = "../datasets/11/train_test/eval0819.jsonl"
model_path = "../models/Qwen2-0.5B-Instruct"
output_path = "../models/Qwen2-0.5B-Instruct_fr_0819"

In [14]:
model_id = "Qwen/Qwen2-0.5B-Instruct"
print("开始从 modelscope下载模型")

snapshot_download(
    model_id=model_id,
    local_dir=model_path,
    # ModelScope 默认就是下载实体文件，不需要特别指定 symlinks 参数
)


开始从 modelscope下载模型


2026-05-23 14:28:24,444 - modelscope - INFO - Target directory already exists, skipping creation.


'../models/Qwen2-0.5B-Instruct'

In [15]:
def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as file:
        data = [json.loads(line) for line in file]
    return pd.DataFrame(data)


In [16]:
tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=False, trust_remote_code=True)
tokenizer

Qwen2Tokenizer(name_or_path='../models/Qwen2-0.5B-Instruct', vocab_size=151643, model_max_length=32768, is_fast=False, padding_side='right', truncation_side='right', special_tokens={'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>', 'additional_special_tokens': ['<|im_start|>', '<|im_end|>']}, clean_up_tokenization_spaces=False),  added_tokens_decoder={
	151643: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151644: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151645: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}

In [17]:
tokenizer("你是谁")

{'input_ids': [105043, 100165], 'attention_mask': [1, 1]}

In [18]:
def preprocess(item, tokenizer, max_length=2048, instruction=None):
    system_message = "You are a helpful assistant."
    instruction = item["instruction"] if instruction is None else instruction
    user_message = instruction + "\n" + item["input"]
    assistant_message = json.dumps({"is_fraud": item["label"]}, ensure_ascii=False)

    message = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message},
        {"role": "assistant", "content": assistant_message},
    ]
    # apply Chat template它只会返回input ID
    full_ids = tokenizer.apply_chat_template(
        message,
        tokenize=True,
        add_generation_prompt=False,
        max_length=max_length,
        truncation=True,
    )
    prompt_ids = tokenizer.apply_chat_template(
        message[:-1],
        tokenize=True,
        add_generation_prompt=True,
        max_length=max_length,
        truncation=True,
    )

    labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]
    return {
        "input_ids": full_ids,
        "attention_mask": [1] * len(full_ids),
        "labels": labels,
    }


In [ ]:
def load_dataset(train_path, eval_path, tokenizer):
    train_df = load_jsonl(train_path)
    train_ds = Dataset.from_pandas(train_df)
    train_dataset = train_ds.map(
        lambda x: preprocess(x, tokenizer),
        remove_columns=train_ds.column_names,
        desc="Tokenizing train dataset",
    )

    eval_df = load_jsonl(eval_path)
    eval_ds = Dataset.from_pandas(eval_df)
    eval_dataset = eval_ds.map(
        lambda x: preprocess(x, tokenizer),
        remove_columns=eval_ds.column_names,
        desc="Tokenizing eval dataset",
    )

    return train_dataset, eval_dataset

: 

In [ ]:
train_dataset, eval_dataset = load_dataset(traindata_path, evaldata_path, tokenizer)